# **Feed Forward Network (FFN)**

## What is the Feed-Forward Network (FFN)?

Self-attention lets words talk to each other — it's about relationships. But after that conversation, each word needs time to think individually — to process and digest what it just learned. That's what FFN does.

Think of it like a classroom:
```
Attention = group discussion (students share information)
FFN       = individual thinking (each student processes what they heard)
```

The FFN is applied to each token independently — same formula, same weights, but each word goes through it separately. It doesn't look at other words. That's the key difference from attention.

The formula is simple:
```
FFN(x) = ReLU(x @ W1 + b1) @ W2 + b2
```

That's just two linear layers with a ReLU activation in between.

```
Step 1: x @ W1 + b1        → expand to bigger dimension
Step 2: ReLU()              → kill negative values (non-linearity!)
Step 3: result @ W2 + b2   → compress back to original dimension
```

Why expand then compress?
The standard pattern is to expand to `4x the size`, then compress back:
`d_model = 8  →  expand to 32 (8 × 4)  →  compress back to 8`

```
Like taking a deep breath:
  inhale (expand): give the model MORE space to think, more neurons to work with
  exhale (compress): squeeze the important stuff back into original size
```

This expansion gives the network a bigger "workspace" to compute complex transformations. In real models:

```
GPT-2: d_model=768   → expand to 3072 (768 × 4) → back to 768
GPT-3: d_model=12288 → expand to 49152           → back to 12288
```

Why ReLU?
To solve the `linearity problem`: Two linear layers stacked without activation is just one linear layer — useless stacking. ReLU breaks linearity so the network can learn complex patterns. Without it:
```
(x @ W1) @ W2 = x @ (W1 @ W2) = x @ W_combined
                                  ↑
                        Just ONE linear layer! No benefit from two.
                        
With ReLU between them: can't simplify, genuinely two layers, more power.
```


## What ReLU actually does?

In [ ]:
# Let's see ReLU in action
import torch

sample = torch.tensor([-2.0, -0.5, 0.0, 0.3, 1.5])
print("Before ReLU:", sample)
print("After ReLU: ", torch.relu(sample))
# [-2.0, -0.5, 0.0, 0.3, 1.5]  →  [0.0, 0.0, 0.0, 0.3, 1.5]
# Negatives become zero, positives stay unchanged

Before ReLU: tensor([-2.0000, -0.5000,  0.0000,  0.3000,  1.5000])
After ReLU:  tensor([0.0000, 0.0000, 0.0000, 0.3000, 1.5000])


# FFN - The Manual Way

In [ ]:
import torch
import math

torch.manual_seed(42)

# Fake input: 7 tokens, d_model=8 (pretend this came from attention)
x = torch.randn(7, 8)

d_model = 8
d_ff = 32   # expand to 4x (8 × 4 = 32)

# Create weight matrices manually
W1 = torch.randn(d_model, d_ff)    # (8, 32) — expand
b1 = torch.zeros(d_ff)             # (32,)

W2 = torch.randn(d_ff, d_model)    # (32, 8) — compress back
b2 = torch.zeros(d_model)          # (8,)

# Forward pass — step by step
print("Input shape:", x.shape)                     # (7, 8)

# Step 1: First linear layer (expand)
hidden = x @ W1 + b1
print("After expand:", hidden.shape)                # (7, 32)

# Step 2: ReLU activation (kill negatives)
hidden = torch.relu(hidden)
print("After ReLU:", hidden.shape)                  # (7, 32)
# Some values are now 0 (the negative ones died)

# Step 3: Second linear layer (compress back)
output = hidden @ W2 + b2
print("After compress:", output.shape)              # (7, 8)

print("\nInput shape == Output shape?", x.shape == output.shape)  # True!

Input shape: torch.Size([7, 8])
After expand: torch.Size([7, 32])
After ReLU: torch.Size([7, 32])
After compress: torch.Size([7, 8])

Input shape == Output shape? True


## FFN - PyTorch Way

Same math as the manual version, but PyTorch manages everything. Compare the two:

```
Manual:                          PyTorch:
W1 = torch.randn(8, 32)      →    self.layer1 = nn.Linear(8, 32)
b1 = torch.zeros(32)         →    (bias included automatically)
hidden = x @ W1 + b1         →    x = self.layer1(x)
hidden = torch.relu(hidden)  →    x = self.relu(x)
W2 = torch.randn(32, 8)      →    self.layer2 = nn.Linear(32, 8)
output = hidden @ W2 + b2    →    x = self.layer2(x)
```


In [ ]:
import torch.nn as nn

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        # Two linear layers — expand then compress
        # nn.Linear handles weights AND biases automatically
        self.layer1 = nn.Linear(d_model, d_ff)       # (8 → 32) expand
        self.relu = nn.ReLU()                        # activation
        self.layer2 = nn.Linear(d_ff, d_model)       # (32 → 8) compress

    def forward(self, x):
        # Each step on its own line for clarity
        x = self.layer1(x)    # expand:   (7, 8)  → (7, 32)
        x = self.relu(x)      # activate: (7, 32) → (7, 32) (negatives → 0)
        x = self.layer2(x)    # compress: (7, 32) → (7, 8)
        return x

# Create and run
ffn = FeedForward(d_model=8, d_ff=32)
x = torch.randn(7, 8)
output = ffn(x)

print("Input shape:", x.shape)       # (7, 8)
print("Output shape:", output.shape)  # (7, 8)

# Let's see how many parameters this has
total_params = sum(p.numel() for p in ffn.parameters())
print(f"\nTotal parameters: {total_params}")
# W1: 8×32=256, b1: 32, W2: 32×8=256, b2: 8 → total = 552

Input shape: torch.Size([7, 8])
Output shape: torch.Size([7, 8])

Total parameters: 552


## How FFN Works?

### Tokens and Neurons:

The 7 tokens are NOT 7 neurons. Tokens and neurons are different things.
```
Tokens = your DATA (the 7 words flowing through the network)
Neurons = the MACHINERY (the processing units inside each layer)
```
Think of it like a factory:
```
Tokens  = 7 items on a conveyor belt (they move through)
Neurons = the machines in the factory (they stay fixed, process each item)
```
The number of neurons in a layer is determined by the output dimension, not by how many tokens you have. Whether you feed 7 tokens or 700 tokens, the FFN has the same number of neurons.

---

### Let's build up from a single neuron
One neuron does this:
```
output = activation(input₁×w₁ + input₂×w₂ + ... + input₈×w₈ + bias)
```
It takes ALL inputs (8 numbers), multiplies each by its own weight, sums them up, adds bias, applies activation. One neuron produces one number.

Let's say our first token "i" has embedding `[0.2, 0.5, 0.1, 0.8, 0.3, 0.7, 0.4, 0.6]` — that's 8 numbers.

One neuron processing this token:
```
neuron_1 output = ReLU(0.2×w₁ + 0.5×w₂ + 0.1×w₃ + 0.8×w₄
                     + 0.3×w₅ + 0.7×w₆ + 0.4×w₇ + 0.6×w₈ + bias)
               = ReLU(some single number)
               = one number
```               
But we need 32 output numbers (expanding to d_ff=32). So we need 32 neurons, each with its own set of 8 weights:
```
neuron_1:  ReLU(0.2×w₁₁ + 0.5×w₁₂ + ... + 0.6×w₁₈ + b₁) = out₁
neuron_2:  ReLU(0.2×w₂₁ + 0.5×w₂₂ + ... + 0.6×w₂₈ + b₂) = out₂
neuron_3:  ReLU(0.2×w₃₁ + 0.5×w₃₂ + ... + 0.6×w₃₈ + b₃) = out₃
...
neuron_32: ReLU(0.2×w₃₂,₁ + 0.5×w₃₂,₂ + ... + 0.6×w₃₂,₈ + b₃₂) = out₃₂
```
32 neurons, each producing one number → output is 32 numbers. That's our expansion from 8 to 32.

---

### 32 Neurons with Inputs:

Let's write all 32 neurons side by side. The weights of all neurons form a matrix:
```
W1 = [ neuron_1 weights:  w₁₁, w₁₂, w₁₃, w₁₄, w₁₅, w₁₆, w₁₇, w₁₈ ]
     [ neuron_2 weights:  w₂₁, w₂₂, w₂₃, w₂₄, w₂₅, w₂₆, w₂₇, w₂₈ ]
     [ neuron_3 weights:  w₃₁, w₃₂, w₃₃, w₃₄, w₃₅, w₃₆, w₃₇, w₃₈ ]
     ...
     [ neuron_32 weights: w₃₂,₁ ...                         w₃₂,₈]

Shape: (32, 8) — 32 neurons, each with 8 weights
```

And the bias of all neurons forms a vector:
```
b1 = [b₁, b₂, b₃, ... b₃₂]

Shape: (32,)
```
Now when we compute x @ W1.T + b1 for one token:
```
input:  [0.2, 0.5, 0.1, 0.8, 0.3, 0.7, 0.4, 0.6]    shape: (8,)
W1.T:   (8, 32)     ← transposed so dimensions align
result: (32,)        ← one output per neuron

input @ W1.T = [
    0.2×w₁₁ + 0.5×w₁₂ + ... + 0.6×w₁₈,    ← neuron 1's sum
    0.2×w₂₁ + 0.5×w₂₂ + ... + 0.6×w₂₈,    ← neuron 2's sum
    ...
    0.2×w₃₂,₁ + 0.5×w₃₂,₂ + ... + 0.6×w₃₂,₈  ← neuron 32's sum
]

+ b1 = add each neuron's bias

ReLU() = apply activation to each neuron's output
```
That's it. Matrix multiplication is just running all neurons simultaneously. Each row of W1 is one neuron's weights. The matrix multiply computes all 32 neurons in one shot.
```
Writing 32 neurons individually:
  neuron_k output = ReLU(input₁×wₖ₁ + input₂×wₖ₂ + ... + input₈×wₖ₈ + bₖ)

Writing them all at once (matrix form):
  all outputs = ReLU(x @ W1.T + b1)

SAME MATH. Matrix form is just the shorthand.
```

---

### Now what about 7 tokens?
We have 7 tokens, each with 8 dimensions. We stack them into a matrix:
```
x = [ token_0: [0.2, 0.5, 0.1, 0.8, 0.3, 0.7, 0.4, 0.6] ]
    [ token_1: [0.9, 0.1, 0.4, 0.3, 0.8, 0.2, 0.5, 0.7] ]
    [ token_2: [0.4, 0.6, 0.8, 0.1, 0.5, 0.9, 0.2, 0.3] ]
    ...
    [ token_6: [0.7, 0.3, 0.6, 0.5, 0.1, 0.4, 0.8, 0.2] ]

Shape: (7, 8)
When we compute x @ W1.T + b1:
(7, 8) @ (8, 32) + (32,) = (7, 32)

Row 0: token_0 goes through all 32 neurons → 32 outputs
Row 1: token_1 goes through all 32 neurons → 32 outputs
Row 2: token_2 goes through all 32 neurons → 32 outputs
...
Row 6: token_6 goes through all 32 neurons → 32 outputs
```
Every token goes through the SAME 32 neurons independently. The matrix multiplication handles all 7 tokens at once, but each token is processed by the same set of neurons with the same weights. Token 0 doesn't affect token 1's FFN computation.

---

### The full FFN with neuron mapping

Now let's trace the complete `FFN(x) = ReLU(x @ W1 + b1) @ W2 + b2` and map it to neurons:
```
LAYER 1: 32 neurons (expanding)
┌───────────────────────────────────────────────────┐
│ Each of the 32 neurons takes 8 inputs             │
│ Each neuron has 8 weights + 1 bias                │
│ All weights together = W1 matrix (32, 8)          │
│ All biases together = b1 vector (32,)             │
│                                                   │
│ For each token independently:                     │
│   step 1: x @ W1.T + b1    (compute all 32 sums)  │
│   step 2: ReLU()            (kill negatives)      │
│   output: 32 numbers per token                    │
│                                                   │
│ input shape:  (7, 8)                              │
│ output shape: (7, 32)                             │
└───────────────────────────────────────────────────┘
                    ↓

LAYER 2: 8 neurons (compressing)
┌──────────────────────────────────────────────────┐
│ Each of the 8 neurons takes 32 inputs            │
│ Each neuron has 32 weights + 1 bias              │
│ All weights together = W2 matrix (8, 32)         │
│ All biases together = b2 vector (8,)             │
│                                                  │
│ For each token independently:                    │
│   result @ W2.T + b2   (compute all 8 sums)      │
│   No activation here (output goes to next layer) │
│   output: 8 numbers per token                    │
│                                                  │
│ input shape:  (7, 32)                            │
│ output shape: (7, 8)                             │
└──────────────────────────────────────────────────┘
```
Summary of the neuron structure:
```
Layer 1: 32 neurons, each takes 8 inputs  → total weights: 32 × 8 = 256
Layer 2: 8 neurons,  each takes 32 inputs → total weights: 8 × 32 = 256
Biases:  32 + 8 = 40
Total parameters: 256 + 256 + 40 = 552
```

---

### Connecting the two formulas:
```
Single neuron formula:
  output = activation(input × weight + bias)

FFN matrix formula:
  FFN(x) = ReLU(x @ W1 + b1) @ W2 + b2

The connection:
  x @ W1 + b1     = running ALL layer-1 neurons at once
  ReLU(...)        = applying activation to all neurons at once  
  ... @ W2 + b2   = running ALL layer-2 neurons at once

Matrix multiplication IS neurons running in parallel.
The formulas are identical — one is zoomed into a single neuron,
the other is zoomed out to see all neurons at once.
```

## FFN - With MultiHead Attention

### Residual Connection (the skip connection):
```
output = layer(x) + x
                    ↑
              ADD THE ORIGINAL INPUT BACK

Why? Deep networks have a problem — as information flows through
many layers, it can degrade or vanish. The residual connection
creates a "shortcut highway" where the original signal can flow
through unchanged. The layer only needs to learn what to ADD
to the input, not recreate everything from scratch.

Think of it like editing a document:
  Without residual: rewrite the entire document from memory each time
  With residual:    keep the original, just write corrections on top
```

### Layer Normalization:

```
output = LayerNorm(x)

What: normalizes each token's vector to have mean≈0 and variance≈1
Why:  keeps numbers in a stable range as they flow through many layers

Without it, after 96 layers, numbers could explode to millions
or shrink to near-zero. LayerNorm resets the scale at each step.

Think of it like recalibrating a scale between each measurement.
```

In [ ]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        # d_k is CALCULATED, not chosen by us
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # each head's dimension

        # Each head needs its own Q, K, V projections
        # But instead of making separate nn.Linear for each head,
        # we use ONE big linear layer and then SPLIT the output
        # This is more efficient — one big matrix multiply instead of many small ones
        self.W_Q = nn.Linear(d_model, d_model, bias=False)  # (8, 8) not (8, 4)!
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)

        # After concatenating all heads, project back to d_model
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        seq_len = x.shape[0]     # number of tokens (7)

        # Step 1: Project into Q, K, V (full d_model size)
        Q = self.W_Q(x)   # (7, 8)
        K = self.W_K(x)   # (7, 8)
        V = self.W_V(x)   # (7, 8)

        # Step 2: SPLIT into multiple heads
        # Reshape from (7, 8) → (7, 2, 4) → (2, 7, 4)
        # 7 tokens, 2 heads, 4 dimensions per head
        Q = Q.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        K = K.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        V = V.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        # Now Q, K, V are each (2, 7, 4) — 2 heads, 7 tokens, 4 dims per head

        # Step 3: Attention (same formula, now per head)
        scores = Q @ K.transpose(-2, -1)       # (2, 7, 7) — each head has its own 7×7
        scaled_scores = scores / math.sqrt(self.d_k)
        attention_weights = torch.softmax(scaled_scores, dim=-1)  # (2, 7, 7)

        # Step 4: Multiply by V
        head_outputs = attention_weights @ V    # (2, 7, 4)

        # Step 5: CONCATENATE all heads back together
        # (2, 7, 4) → (7, 2, 4) → (7, 8)
        head_outputs = head_outputs.permute(1, 0, 2).contiguous().view(seq_len, self.d_model)

        # Step 6: Final projection
        output = self.W_O(head_outputs)    # (7, 8)

        return output, attention_weights


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        # Two linear layers — expand then compress
        # nn.Linear handles weights AND biases automatically
        self.layer1 = nn.Linear(d_model, d_ff)       # (8 → 32) expand
        self.relu = nn.ReLU()                        # activation
        self.layer2 = nn.Linear(d_ff, d_model)       # (32 → 8) compress

    def forward(self, x):
        # Each step on its own line for clarity
        x = self.layer1(x)    # expand:   (7, 8)  → (7, 32)
        x = self.relu(x)      # activate: (7, 32) → (7, 32) (negatives → 0)
        x = self.layer2(x)    # compress: (7, 32) → (7, 8)
        return x


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        # The two main components
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff)

        # Layer normalization (stabilizes training)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # === ATTENTION BLOCK ===
        # 1. Normalize
        normed = self.norm1(x)
        # 2. Multi-head attention
        attn_output, attn_weights = self.attention(normed)
        # 3. Residual connection (add original input back)
        x = x + attn_output

        # === FFN BLOCK ===
        # 4. Normalize
        normed = self.norm2(x)
        # 5. Feed-forward
        ff_output = self.feed_forward(normed)
        # 6. Residual connection (add input back again)
        x = x + ff_output

        return x, attn_weights

Each sub-block follows: Normalize → Process → Add back original. This pattern repeats throughout the entire Transformer.

```
Input
  ↓
  ├──────────────────────┐
  ↓                      │
  LayerNorm              │  Residual
  ↓                      │  Connection
  Multi-Head Attention   │
  ↓                      │
  + ←────────────────────┘  (add original back)
  ↓
  ├──────────────────────┐
  ↓                      │
  LayerNorm              │  Residual
  ↓                      │  Connection
  Feed-Forward           │
  ↓                      │
  + ←────────────────────┘  (add original back)
  ↓
Output
```

### Run it:

In [ ]:
# Full pipeline: Text → Tokens → Embeddings → Transformer Block

# Sentence setup
text = "I love code and I love AI"
tokens = text.lower().split()
vocab = sorted(set(tokens))
word_to_id = {word: idx for idx, word in enumerate(vocab)}
token_ids = torch.tensor([word_to_id[t] for t in tokens])

# Config
d_model = 8
num_heads = 2
d_ff = 32       # 4 × d_model

# Embedding + Positional Encoding
embedding = nn.Embedding(len(vocab), d_model)

# Create positional encoding
def create_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    for pos in range(max_len):
        for i in range(0, d_model, 2):
            denominator = 10000 ** (i / d_model)
            pe[pos, i]     = math.sin(pos / denominator)
            pe[pos, i + 1] = math.cos(pos / denominator)
    return pe

pe = create_positional_encoding(10, d_model)
final_embeddings = embedding(token_ids) + pe[:len(tokens)]

# Create and run the Transformer block
transformer_block = TransformerBlock(d_model=8, num_heads=2, d_ff=32)
output, attention_weights = transformer_block(final_embeddings)

print("=== FULL TRANSFORMER BLOCK ===")
print(f"Input shape:  {final_embeddings.shape}")   # (7, 8)
print(f"Output shape: {output.shape}")              # (7, 8)
print(f"Attention:    {attention_weights.shape}")    # (2, 7, 7)

# Count ALL parameters
total = sum(p.numel() for p in transformer_block.parameters())
print(f"\nTotal trainable parameters: {total}")

print("\n=== Layer by layer breakdown ===")
for name, param in transformer_block.named_parameters():
    print(f"  {name}: {param.shape}")

=== FULL TRANSFORMER BLOCK ===
Input shape:  torch.Size([7, 8])
Output shape: torch.Size([7, 8])
Attention:    torch.Size([2, 7, 7])

Total trainable parameters: 840

=== Layer by layer breakdown ===
  attention.W_Q.weight: torch.Size([8, 8])
  attention.W_K.weight: torch.Size([8, 8])
  attention.W_V.weight: torch.Size([8, 8])
  attention.W_O.weight: torch.Size([8, 8])
  feed_forward.layer1.weight: torch.Size([32, 8])
  feed_forward.layer1.bias: torch.Size([32])
  feed_forward.layer2.weight: torch.Size([8, 32])
  feed_forward.layer2.bias: torch.Size([8])
  norm1.weight: torch.Size([8])
  norm1.bias: torch.Size([8])
  norm2.weight: torch.Size([8])
  norm2.bias: torch.Size([8])


## Flow of Execution:

```
INPUT
"I love code and I love AI"
token_ids = [3, 4, 2, 1, 3, 4, 0]
shape: (7,)  ← just 7 integers
        │
        ▼
┌─────────────────────────────┐
│     TOKEN EMBEDDING         │
│  nn.Embedding(5, 8)         │
│                             │
│  Look up row 3 → "i"        │
│  Look up row 4 → "love"     │
│  ...                        │
│                             │
│  (7,) → (7, 8)              │
└─────────────────────────────┘
        │
        ▼
┌─────────────────────────────┐
│   POSITIONAL ENCODING       │
│   (fixed sin/cos table)     │
│                             │
│   pe[:7] shape: (7, 8)      │
│                             │
│   final = token_emb + pe    │
│   (7, 8) + (7, 8) = (7, 8)  │
└─────────────────────────────┘
        │
        ▼
════════════════════════════════
   TRANSFORMER BLOCK STARTS
════════════════════════════════
        │
        ├────────────────────────────────────────┐
        │                                        │ RESIDUAL
        ▼                                        │ (save original)
┌─────────────────────────────┐                  │
│      LAYER NORM 1           │                  │
│   norm1(x)                  │                  │
│                             │                  │
│   Each token's 8 numbers    │                  │
│   rescaled to mean≈0, var≈1 │                  │
│                             │                  │
│   (7, 8) → (7, 8)           │                  │
└─────────────────────────────┘                  │
        │                                        │
        ▼                                        │
┌─────────────────────────────┐                  │
│   MULTI-HEAD ATTENTION      │                  │
│                             │                  │
│  ┌──────────────────────┐   │                  │
│  │ W_Q, W_K, W_V        │   │                  │
│  │ nn.Linear(8, 8)      │   │                  │
│  │                      │   │                  │
│  │ Q = x @ W_Q          │   │                  │
│  │ K = x @ W_K          │   │                  │
│  │ V = x @ W_V          │   │                  │
│  │ (7,8)→(7,8) each     │   │                  │
│  └──────────────────────┘   │                  │
│            │                │                  │
│            ▼                │                  │
│  ┌──────────────────────┐   │                  │
│  │ SPLIT INTO 2 HEADS   │   │                  │
│  │                      │   │                  │
│  │ (7,8) → (7,2,4)      │   │                  │
│  │      → (2,7,4)       │   │                  │
│  │                      │   │                  │
│  │ Head1: (7,4) queries │   │                  │
│  │ Head2: (7,4) queries │   │                  │
│  └──────────────────────┘   │                  │
│            │                │                  │
│            ▼                │                  │
│  ┌──────────────────────┐   │                  │
│  │ ATTENTION per head   │   │                  │
│  │                      │   │                  │
│  │ scores = Q @ K.T     │   │                  │
│  │ (2,7,4)@(2,4,7)      │   │                  │
│  │ = (2,7,7)            │   │                  │
│  │                      │   │                  │
│  │ ÷ √d_k               │   │                  │
│  │                      │   │                  │
│  │ softmax → (2,7,7)    │   │                  │
│  │                      │   │                  │
│  │ @ V → (2,7,4)        │   │                  │
│  └──────────────────────┘   │                  │
│            │                │                  │
│            ▼                │                  │
│  ┌──────────────────────┐   │                  │
│  │ CONCATENATE HEADS    │   │                  │
│  │                      │   │                  │
│  │ (2,7,4) → (7,2,4)    │   │                  │
│  │        → (7,8)       │   │                  │
│  └──────────────────────┘   │                  │
│            │                │                  │
│            ▼                │                  │
│  ┌──────────────────────┐   │                  │
│  │ OUTPUT PROJECTION    │   │                  │
│  │ W_O: nn.Linear(8,8)  │   │                  │
│  │ (7,8) → (7,8)        │   │                  │
│  └──────────────────────┘   │                  │
│                             │                  │
│   output shape: (7, 8)      │                  │
└─────────────────────────────┘                  │
        │                                        │
        ▼                                        │
        + ◄──────────────────────────────────────┘
        │  ADD ORIGINAL INPUT BACK (residual)
        │  (7,8) + (7,8) = (7,8)
        │
        ├────────────────────────────────────────┐
        │                                        │ RESIDUAL
        ▼                                        │ (save this x)
┌─────────────────────────────┐                  │
│      LAYER NORM 2           │                  │
│   norm2(x)                  │                  │
│   (7, 8) → (7, 8)           │                  │
└─────────────────────────────┘                  │
        │                                        │
        ▼                                        │
┌─────────────────────────────┐                  │
│    FEED FORWARD NETWORK     │                  │
│                             │                  │
│  layer1: nn.Linear(8, 32)   │                  │
│  (7,8) → (7,32)   [expand]  │                  │
│                             │                  │
│  ReLU()                     │                  │
│  (7,32) → (7,32)            │                  │
│  [negatives → 0]            │                  │
│                             │                  │
│  layer2: nn.Linear(32, 8)   │                  │
│  (7,32) → (7,8)  [compress] │                  │
│                             │                  │
│  output shape: (7, 8)       │                  │
└─────────────────────────────┘                  │
        │                                        │
        ▼                                        │
        + ◄──────────────────────────────────────┘
        │  ADD INPUT BACK AGAIN (residual)
        │  (7,8) + (7,8) = (7,8)
        │
        ▼
════════════════════════════════
   TRANSFORMER BLOCK OUTPUT
   shape: (7, 8)
════════════════════════════════

Input shape:  (7, 8)
Output shape: (7, 8)  ← same! ready for next block
```

### What LayerNorm does:

Take one token's vector after attention — say token "love":
```
"love" vector: [4.2, -1.8, 0.3, 6.7, -2.1, 0.8, 3.5, -0.4]
```
These numbers could be wildly different scales. LayerNorm normalizes them — rescales so they have mean=0 and variance=1:
```
Step 1: Calculate mean
  mean = (4.2 + -1.8 + 0.3 + 6.7 + -2.1 + 0.8 + 3.5 + -0.4) / 8
       = 11.2 / 8 = 1.4

Step 2: Calculate variance (how spread out the numbers are)
  variance = average of (each value - mean)²
           = some number, let's say 7.2

Step 3: Normalize
  normalized = (original - mean) / √variance
  
  "love" normalized: [(4.2-1.4)/√7.2, (-1.8-1.4)/√7.2, ...]
                    = [1.04, -1.19, -0.41, 1.97, -1.31, ...]
  
  Now mean ≈ 0, variance ≈ 1  ✓
```  
This is done per token — each token's 8 numbers get normalized independently. This keeps numbers in a stable range as they flow through many layers.

But here's the problem: pure normalization is too aggressive. It erases learned information. If the model spent millions of training steps learning that "love" should have large values in dimension 3, normalization just wiped that out.

Solution — learnable scale and shift (the norm weights and bias):

After normalizing, LayerNorm applies one more step:
```
output = (normalized × gamma) + beta

gamma = norm weight  (learned scale)
beta  = norm bias    (learned shift)
```
These are learnable parameters — they start as:
```
gamma (weight) = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]  ← all ones
beta  (bias)   = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]  ← all zeros
```
At the start, multiplying by 1 and adding 0 changes nothing — normalization is fully applied. But during training, gamma and beta get adjusted. The model learns:
```
"Dimension 3 is really important — let me scale it up"
  → gamma[3] becomes 2.5 instead of 1.0

"Dimension 6 needs to be shifted higher"
  → beta[6] becomes 0.8 instead of 0.0
```  
So LayerNorm is actually three steps:
```
Step 1: Normalize (mean=0, var=1)  — removes wild scale differences
Step 2: × gamma                    — learned rescaling per dimension
Step 3: + beta                     — learned shift per dimension
```
The model controls HOW MUCH normalization to apply
via gamma and beta.